# 🎬 MoneyPrinterTurbo Panel — Colab

Bu notebook MPT'yi Colab'de kurar ve sana **tıklanabilir panel linki** verir.

## Kullanım
1. Üstteki ▶️ düğmesiyle hücreleri **sırayla** çalıştır (1 → 2 → 3).
2. 3. hücrenin sonunda çıkan `https://....trycloudflare.com` linkine tıkla → panel açılır.
3. Paneli kapatmak istersen Colab sekmesini kapat; link de ölür. Tekrar açmak için notebook'u yeniden çalıştır.

## Anahtarlar (Colab Secrets)
Soldaki 🔑 (anahtar) simgesinden şu isimlerle ekle:
| İsim | Nereden |
|---|---|
| `PIXABAY_API_KEY` | https://pixabay.com/api/docs/ |
| `GEMINI_API_KEY` | https://aistudio.google.com/app/apikey |
| `ELEVENLABS_API_KEY` | https://elevenlabs.io/app/settings/api-keys (isteğe bağlı) |

> Not: MPT Bilgi Dozu şablonunu (üst başlık + karaoke altyazı + TAKİP ET) basmaz. Bu panel **hızlı/deneme** videosu için; kanalın kendi motoru ayrı.


In [ ]:
#@title 1) Kurulum (2-5 dk)
!nvidia-smi -L || echo "GPU yok, CPU ile devam"
%cd /content
!rm -rf MoneyPrinterTurbo
!git clone --depth 1 https://github.com/harry0703/MoneyPrinterTurbo.git
%cd MoneyPrinterTurbo
!apt-get -qq update && apt-get -qq install -y ffmpeg imagemagick > /dev/null
!pip install -q -r requirements.txt
print("KURULUM TAMAM")

In [ ]:
#@title 2) Ayarlar + anahtarlar
import pathlib, textwrap

def secret(ad, varsayilan=""):
    try:
        from google.colab import userdata
        return userdata.get(ad) or varsayilan
    except Exception:
        return varsayilan

pexels  = secret("PEXELS_API_KEY")
pixabay = secret("PIXABAY_API_KEY")
gemini  = secret("GEMINI_API_KEY")
eleven  = secret("ELEVENLABS_API_KEY")

if not (pixabay or pexels):
    print("⚠️ PIXABAY_API_KEY veya PEXELS_API_KEY yok — panel açılır ama video için görsel kaynağı bulamaz.")

cfg = textwrap.dedent(f'''
[app]
llm_provider = "gemini"
video_source = "pixabay"
subtitle_provider = "edge"
hide_config = false

pexels_api_keys = ["{pexels}"]
pixabay_api_keys = ["{pixabay}"]

gemini_api_key = "{gemini}"
gemini_model_name = ""

elevenlabs_api_key = "{eleven}"

[elevenlabs]
api_key = "{eleven}"
model_id = "eleven_multilingual_v2"

[ui]
hide_log = false
''')

pathlib.Path("/content/MoneyPrinterTurbo/config.toml").write_text(cfg, encoding="utf-8")
print("config.toml yazıldı. Anahtarlar:",
      {"pixabay": bool(pixabay), "pexels": bool(pexels), "gemini": bool(gemini), "elevenlabs": bool(eleven)})

In [ ]:
#@title 3) Paneli başlat + link al
import subprocess, time, re, pathlib, os

# eski süreçleri kapat
!pkill -f streamlit || true
!pkill -f cloudflared || true

%cd /content/MoneyPrinterTurbo
log = open("/content/streamlit.log", "w")
subprocess.Popen(
    "streamlit run webui/Main.py --server.port 8501 --server.address 0.0.0.0 "
    "--server.headless true --server.enableCORS false --browser.gatherUsageStats false",
    shell=True, stdout=log, stderr=subprocess.STDOUT)

if not pathlib.Path("/content/cloudflared").exists():
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared

cflog = open("/content/cf.log", "w")
subprocess.Popen("/content/cloudflared tunnel --url http://localhost:8501 --no-autoupdate",
                 shell=True, stdout=cflog, stderr=subprocess.STDOUT)

link = None
for _ in range(60):
    time.sleep(2)
    metin = pathlib.Path("/content/cf.log").read_text(errors="ignore")
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", metin)
    if m:
        link = m.group(0)
        break

if link:
    print("=" * 60)
    print("PANEL HAZIR 👉", link)
    print("=" * 60)
    print("Link ~birkaç saat yaşar; Colab sekmesi kapanınca ölür.")
else:
    print("Link alınamadı. /content/cf.log dosyasına bak:")
    print(pathlib.Path("/content/cf.log").read_text(errors="ignore")[-1500:])